# AWS Bedrock - Amazon Nova Reel Video Generation

Generate a video using Amazon Nova Reel (`amazon.nova-reel-v1:0`) via AWS Bedrock async invocation. Output is stored in S3.

In [1]:
import boto3
import json
import time

# ── Configuration ──────────────────────────────────────────────────────────────
AWS_REGION   = "us-east-1"          # Nova Reel is available in us-east-1
MODEL_ID     = "amazon.nova-reel-v1:0"
PROMPT       = "A person dancing on a mountain"
S3_BUCKET    = "gen-ai-exercise-pappuk"   # <-- replace with your bucket
S3_PREFIX    = "nova-reel-output"

bedrock = boto3.client("bedrock-runtime", region_name=AWS_REGION)
print(f"Bedrock client ready | region: {AWS_REGION}")

Bedrock client ready | region: us-east-1


In [2]:
# ── Start async video generation ───────────────────────────────────────────────
model_input = {
    "taskType": "TEXT_VIDEO",
    "textToVideoParams": {
        "text": PROMPT
    },
    "videoGenerationConfig": {
        "durationSeconds": 6,   # 6 s is the minimum; max is 120 s
        "fps": 24,
        "dimension": "1280x720",
        "seed": 42
    }
}

response = bedrock.start_async_invoke(
    modelId=MODEL_ID,
    modelInput=model_input,
    outputDataConfig={
        "s3OutputDataConfig": {
            "s3Uri": f"s3://{S3_BUCKET}/{S3_PREFIX}/"
        }
    }
)

invocation_arn = response["invocationArn"]
print(f"Job started.\nInvocation ARN: {invocation_arn}")

Job started.
Invocation ARN: arn:aws:bedrock:us-east-1:874348038830:async-invoke/02ys5s61f4w7


In [3]:
# ── Poll until complete ─────────────────────────────────────────────────────────
POLL_INTERVAL = 30  # seconds between status checks

while True:
    status_response = bedrock.get_async_invoke(invocationArn=invocation_arn)
    status = status_response["status"]
    print(f"[{time.strftime('%H:%M:%S')}] Status: {status}")

    if status == "Completed":
        s3_output_uri = (
            status_response["outputDataConfig"]["s3OutputDataConfig"]["s3Uri"]
        )
        print(f"\nVideo generation complete!")
        print(f"Output S3 URI : {s3_output_uri}")
        print(f"Download via  : aws s3 cp {s3_output_uri} .")
        break
    elif status == "Failed":
        failure_msg = status_response.get("failureMessage", "No details provided.")
        print(f"\nJob failed: {failure_msg}")
        break

    time.sleep(POLL_INTERVAL)

[20:33:30] Status: InProgress
[20:34:00] Status: InProgress
[20:34:30] Status: InProgress
[20:35:01] Status: InProgress
[20:35:31] Status: Completed

Video generation complete!
Output S3 URI : s3://gen-ai-exercise-pappuk/nova-reel-output/02ys5s61f4w7
Download via  : aws s3 cp s3://gen-ai-exercise-pappuk/nova-reel-output/02ys5s61f4w7 .
